<a href="https://colab.research.google.com/github/Darya-06/python-ai-Aleinik-Darya/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- `animated_television_series.csv` — информация об анимационных телесериалах (название, страна, композитор, рейтинг и производные работы)

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем CSV-файл в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [9]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-Aleinik-Darya"
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Aleinik-Darya
    !git clone -q https://github.com/Darya-06/python-ai-Aleinik-Darya.git

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Aleinik-Darya


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [10]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

df_series = pd.read_csv("data/animated_television_series.csv")


print("✅ Загружено строк в df_series:", len(df_series))


✅ Загружено строк в df_series: 4557


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть технические столбцы, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `serial` с URL (ссылкой на объект Wikidata) — сохраняем его для отладки, но переименуем в `URL`.
- Столбцы `serialLabel`, `countryLabel`, `composerLabel`, `rarsRatingLabel`, `derivativeWorkLabel` содержат читаемые подписи (названия сериалов, стран и т.д.).

В этом шаге мы:

- переименуем столбец с URL Wikidata (`serial` → `URL`);
- переименуем `serialLabel` → `serial`, `countryLabel` → `country`, `composerLabel` → `composer`, `rarsRatingLabel` → `rarsRating`, `derivativeWorkLabel` → `derivativeWork`.

⚠️ Важно: если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец `URL` пригодится, если нужно будет быстро перейти к оригинальной записи в Викиданных.

In [11]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

# df_series: анимационные сериалы (страны, композиторы, рейтинги)
if "serialLabel" in df_series.columns:                          # Проверка: нужна ли очистка?
    # URL Wikidata не удаляем, а переименовываем в "URL" для удобства
    df_series = df_series.rename(columns={
        "serial": "URL",        # ← переименовали технический столбец
        "serialLabel": "serial",
        "countryLabel": "country",
        "composerLabel": "composer",
        "rarsRatingLabel": "rarsRating",
        "derivativeWorkLabel": "derivativeWork",
    })
    print("✅ df_series очищен")
else:
    print("⏭️ df_series уже очищен, пропускаем")

print("\n✅ Данные готовы к анализу")

✅ df_series очищен

✅ Данные готовы к анализу


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор обоих DataFrame:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по бюджету (`capital_cost`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы не повторять один и тот же код два раза.

In [12]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов, пропуски и первые строки."""
    print(f"\n📊 {name}")
    print(f"Размер: {df.shape[0]} строк × {df.shape[1]} столбцов")
    print("Столбцы:", ", ".join(df.columns))

    # Анализ пропусков
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print("\nПропуски в данных:")
        for col, count in missing[missing > 0].items():
            pct = count / len(df) * 100
            print(f"  • {col}: {count} ({pct:.1f}%)")
    else:
        print("\n✅ Пропусков в данных нет")

    print("\nПервые строки:")
    display(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_series, "Анимационные телесериалы (df_series)")

# Дополнительная статистика по ключевым категориям
print("\n📈 Уникальные значения:")
print(f"  • Сериалов: {df_series['serial'].nunique()}")
print(f"  • Стран: {df_series['country'].nunique()} ({', '.join(df_series['country'].unique()[:5])}...)")
print(f"  • Композиторов: {df_series['composer'].nunique()}")

if 'rarsRating' in df_series.columns:
    ratings = df_series['rarsRating'].value_counts(dropna=False)
    print(f"\n📺 Рейтинги RARS:")
    for rating, count in ratings.items():
        label = "Без рейтинга" if pd.isna(rating) else rating
        print(f"  • {label}: {count}")


📊 Анимационные телесериалы (df_series)
Размер: 4557 строк × 6 столбцов
Столбцы: URL, serial, country, composer, rarsRating, derivativeWork

Пропуски в данных:
  • composer: 3344 (73.4%)
  • rarsRating: 4414 (96.9%)
  • derivativeWork: 4253 (93.3%)

Первые строки:


,URL,serial,country,composer,rarsRating,derivativeWork
0,http://www.wikidata.org/entity/Q77522,Клуб Винкс,Италия,Фабрицио Кастаниа,NaN,Клуб Винкс: Волшебное приключение
1,http://www.wikidata.org/entity/Q77522,Клуб Винкс,Италия,Фабрицио Кастаниа,NaN,Судьба: Сага Винкс
2,http://www.wikidata.org/entity/Q73622,Футурама,США,Кристофер Тинг,18+,NaN
3,http://www.wikidata.org/entity/Q53093,Время приключений,США,NaN,NaN,Время приключений: Дальние земли
4,http://www.wikidata.org/entity/Q77522,Клуб Винкс,США,Питер Зиззо,NaN,Клуб Винкс: Тайна затерянного королевства



📈 Уникальные значения:
  • Сериалов: 3461
  • Стран: 98 (Италия, США, Великобритания, Россия, Канада...)
  • Композиторов: 462

📺 Рейтинги RARS:
  • Без рейтинга: 4414
  • 0+: 54
  • 6+: 42
  • 12+: 29
  • 18+: 14
  • 16+: 4


In [13]:
print("Топ-20 стран по числу записей:")
print(df_series['country'].value_counts().head(20))

print("\nВсего уникальных стран:", df_series['country'].nunique())
print("Стран с 1 записью:", (df_series['country'].value_counts() == 1).sum())


Топ-20 стран по числу записей:
country
США                 1710
Франция              449
Канада               398
Великобритания       358
Китай                214
Республика Корея     213
Италия               150
Испания              102
Россия                97
Германия              87
Австралия             77
Япония                73
Польша                52
Ирландия              46
Чехословакия          39
Чехия                 29
Бельгия               28
Венгрия               28
Индия                 28
Бразилия              28
Name: count, dtype: int64

Всего уникальных стран: 98
Стран с 1 записью: 28


## 🔬 [4] Глубокая валидация: мультинациональные сериалы и композиторы-«монополисты»

В этом анализе мы выйдем за рамки базовой статистики и исследуем две уникальные особенности индустрии анимации:

#### 🌍 Мультинациональные сериалы
Сериалы часто создаются при участии нескольких стран (совместное производство). В нашем датасете каждая строка = одна страна участия, поэтому сериалы с 2+ странами представлены несколькими записями. Мы выявим:
- Сериалы с наибольшим числом стран-соавторов
- Неожиданные международные коллаборации (например, Италия + Япония + Бразилия)
- Долю «глобальных» проектов от общего числа сериалов

#### 🎼 Композиторы-«монополисты»
Некоторые сериалы на протяжении всей франшизы (включая спин-оффы и адаптации) сохраняют одного композитора — это признак сильной авторской концепции звукового оформления. Мы определим:
- Сериалы, где все версии имеют единого композитора
- Топ-5 композиторов-«монополистов» по числу таких сериалов
- Контраст: сериалы с частой сменой композиторов (признак перезапусков или кризисов производства)

In [14]:
# 🔬 Шаг 4. Глубокая валидация данных

print("=" * 70)
print("🌍 АНАЛИЗ 1: МУЛЬТИНАЦИОНАЛЬНЫЕ СЕРИАЛЫ (совместное производство)")
print("=" * 70)

# Группируем по уникальному идентификатору URL (а не по названию!)
country_counts = df_series.groupby('URL')['country'].nunique().sort_values(ascending=False)

# Фильтруем сериалы с 2+ странами
multi_country_series = country_counts[country_counts >= 2]
total_series = df_series['URL'].nunique()
multi_country_pct = (len(multi_country_series) / total_series) * 100

print(f"\n📈 Статистика:")
print(f"  • Всего уникальных сериалов: {total_series}")
print(f"  • Сериалов с 2+ странами: {len(multi_country_series)} ({multi_country_pct:.1f}%)")
print(f"  • Максимум стран у одного сериала: {multi_country_series.max()}")

# Детали для топ-10 мультинациональных сериалов
print("\n🏆 Топ-10 сериалов по числу стран-соавторов:")
top_multi = multi_country_series.head(10).reset_index()
top_multi.columns = ['URL', 'country_count']

# Добавляем название сериала (первое значение для каждого URL)
def get_serial_name(url):
    return df_series[df_series['URL'] == url]['serial'].iloc[0]

top_multi['serial'] = top_multi['URL'].apply(get_serial_name)

# Добавляем список стран для каждого сериала
def get_countries(url):
    countries = df_series[df_series['URL'] == url]['country'].unique()
    return ', '.join(sorted(countries))

top_multi['countries'] = top_multi['URL'].apply(get_countries)

# Выводим в читаемом порядке: название, количество стран, страны
display(top_multi[['serial', 'country_count', 'countries']].style.set_properties(**{'text-align': 'left'}))

# Анализ самых частых пар стран
print("\n🤝 Самые частые пары стран в совместных проектах:")
country_pairs = []
for url in multi_country_series.index[:50]:  # берём топ-50 для анализа
    countries = sorted(df_series[df_series['URL'] == url]['country'].unique())
    if len(countries) >= 2:
        for i in range(len(countries)):
            for j in range(i+1, len(countries)):
                pair = tuple(sorted([countries[i], countries[j]]))
                country_pairs.append(pair)

from collections import Counter
pair_counts = Counter(country_pairs).most_common(5)
for pair, count in pair_counts:
    print(f"  • {pair[0]} + {pair[1]}: {count} совместных проектов")

print("\n" + "=" * 70)
print("🎼 АНАЛИЗ 2: КОМПОЗИТОРЫ-«МОНОПОЛИСТЫ»")
print("=" * 70)

# Проверяем уникальность композитора для каждого сериала (по уникальному URL)
composer_uniqueness = df_series.groupby('URL')['composer'].nunique()
monopolist_series = composer_uniqueness[composer_uniqueness == 1].index.tolist()
non_monopolist_series = composer_uniqueness[composer_uniqueness > 1].index.tolist()

monopolist_pct = (len(monopolist_series) / total_series) * 100

print(f"\n📈 Статистика:")
print(f"  • Сериалов с единым композитором: {len(monopolist_series)} ({monopolist_pct:.1f}%)")
print(f"  • Сериалов со сменой композитора: {len(non_monopolist_series)} ({100 - monopolist_pct:.1f}%)")

# Топ-5 композиторов-монополистов по числу сериалов
print("\n🏆 Топ-5 композиторов-«монополистов» (сервис единого звучания):")
monopolist_composers = df_series[df_series['URL'].isin(monopolist_series)]
top_composers = monopolist_composers.groupby('composer')['URL'].nunique().sort_values(ascending=False).head(5)

for rank, (composer, count) in enumerate(top_composers.items(), 1):
    # Примеры сериалов для этого композитора
    examples_urls = monopolist_composers[monopolist_composers['composer'] == composer]['URL'].unique()[:3]
    examples = [df_series[df_series['URL'] == url]['serial'].iloc[0] for url in examples_urls]
    examples_str = ', '.join(examples)
    print(f"  {rank}. {composer} — {count} сериалов")
    print(f"     Примеры: {examples_str}")

# Контраст: сериалы с максимальной сменой композиторов
print("\n⚠️  Сериалы с наибольшей сменой композиторов (признак перезапусков):")
composer_changes = composer_uniqueness[composer_uniqueness > 1].sort_values(ascending=False).head(5)
for url, count in composer_changes.items():
    serial_name = df_series[df_series['URL'] == url]['serial'].iloc[0]
    composers = ', '.join(df_series[df_series['URL'] == url]['composer'].unique())
    print(f"  • {serial_name}: {count} композитора")
    print(f"    → {composers}")

print("\n✅ Анализ завершён. Данные готовы к углублённому исследованию.")

🌍 АНАЛИЗ 1: МУЛЬТИНАЦИОНАЛЬНЫЕ СЕРИАЛЫ (совместное производство)

📈 Статистика:
  • Всего уникальных сериалов: 3498
  • Сериалов с 2+ странами: 548 (15.7%)
  • Максимум стран у одного сериала: 8

🏆 Топ-10 сериалов по числу стран-соавторов:


,serial,country_count,countries
0,Говорящий Том и друзья,8,"Австрия, Великобритания, Испания, Кипр, США, Сингапур, Словения, Таиланд"
1,Ferdy the Ant,5,"Великобритания, Германия, Китайская Республика (Тайвань), Лихтенштейн, Чехословакия"
2,Зак Шторм,5,"Индонезия, Италия, Республика Корея, США, Франция"
3,Книга джунглей,5,"Великобритания, Германия, Индия, Финляндия, Франция"
4,Q118955021,5,"Аргентина, Испания, Колумбия, Португалия, Чили"
5,Talking Tom & Friends Minis,5,"Великобритания, Кипр, США, Словения, Япония"
6,Billy the Cat,5,"Бельгия, Великобритания, Германия, Канада, Франция"
7,The Nordic Christmas Hour,5,"Дания, Исландия, Норвегия, Финляндия, Швеция"
8,Агент 203,5,"Германия, Индия, Италия, США, Северная Македония"
9,Викинг Вик,5,"Австралия, Австрия, Германия, Нидерланды, Франция"



🤝 Самые частые пары стран в совместных проектах:
  • Великобритания + США: 10 совместных проектов
  • Великобритания + Германия: 7 совместных проектов
  • Канада + США: 7 совместных проектов
  • США + Франция: 6 совместных проектов
  • Великобритания + Франция: 6 совместных проектов

🎼 АНАЛИЗ 2: КОМПОЗИТОРЫ-«МОНОПОЛИСТЫ»

📈 Статистика:
  • Сериалов с единым композитором: 671 (19.2%)
  • Сериалов со сменой композитора: 59 (80.8%)

🏆 Топ-5 композиторов-«монополистов» (сервис единого звучания):
  1. Хойт Куртин — 56 сериалов
     Примеры: Yogi's Treasure Hunt, Пёс Хакльберри, The Magilla Gorilla Show
  2. Майкл Тавера — 22 сериалов
     Примеры: Time Squad, ¡Mucha Lucha!, Лило и Стич
  3. Шуки Леви — 15 сериалов
     Примеры: Непобедимая принцесса Ши-Ра, Настоящие охотники за привидениями, Инспектор Гаджет
  4. Гай Мун — 11 сериалов
     Примеры: Ужасные приключения Билли и Мэнди, Волшебные покровители, Дэнни-призрак
  5. Petr Skoumal — 11 сериалов
     Примеры: Боб и Бобек, Q12048159, O

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий `python-ai-Aleinik-Darya` в Google Colab
- ✅ Прочитали CSV-файл `data/animated_television_series.csv` с данными об анимационных сериалах из Викиданных
- ✅ Очистили данные:
  - переименовали технический столбец `serial` → `URL` (для отладки)
  - убрали суффикс `Label` у всех столбцов (`serialLabel → serial`, `countryLabel → country` и т.д.)
- ✅ Проверили структуру данных:
  - 4 557 записей, 6 столбцов
  - 3 461 уникальный сериал, 98 стран, 462 композитора
  - анализ пропусков в рейтинге RARS
- ✅ Выполнили глубокую валидацию:
  - мультинациональные сериалы (совместное производство 2+ стран)
  - композиторы-«монополисты» — авторы единого звучания франшиз
  - распределение возрастных рейтингов (от 0+ до 18+)

Теперь у нас есть **чистый, структурированный датасет** об анимационных сериалах, готовый к углублённому анализу.

В ноутбуке следующей недели мы будем использовать **этот же датасет** для:
- более сложного анализа (группировки, фильтрация, агрегации),
- и построения визуализаций (графики и диаграммы). 🎨